In [ ]:
!pip install -q scikit-image matplotlib ezdxf

In [ ]:
import cv2
import json
import numpy as np

from pathlib import Path

from skimage.metrics import structural_similarity as ssim

import matplotlib.pyplot as plt

In [ ]:
# ======================================================
# Load State
# ======================================================


ROOT = Path.cwd().parent


STATE_FILE = (

ROOT /

"config" /

"project_state.json"

)



with open(
    STATE_FILE,
    encoding="utf-8"
) as f:

    PROJECT_STATE=json.load(f)



PROJECT_STATE

In [ ]:
# ======================================================
# Original
# ======================================================


ORIGINAL_PATH = Path(

    PROJECT_STATE["processed_image"]

)



original=cv2.imread(

    str(ORIGINAL_PATH),

    cv2.IMREAD_GRAYSCALE

)



print(

original.shape

)

In [ ]:
# ======================================================
# Load Geometry
# ======================================================


with open(

    PROJECT_STATE["geometry_objects"],

    encoding="utf-8"

) as f:


    GEOMETRY_OBJECTS=json.load(f)



print(

len(GEOMETRY_OBJECTS)

)

In [ ]:
# ======================================================
# Render CAD Objects
# ======================================================


def render_geometry(objects,size):


    canvas=np.zeros(

        size,

        dtype=np.uint8

    )


    for obj in objects:


        if obj["type"]=="LINE":


            cv2.line(

                canvas,

                obj["start"],

                obj["end"],

                255,

                2

            )


        elif obj["type"]=="CIRCLE":


            cv2.circle(

                canvas,

                obj["center"],

                obj["radius"],

                255,

                2

            )


        elif obj["type"]=="POLYLINE":


            pts=np.array(

                obj["points"]

            )


            cv2.polylines(

                canvas,

                [pts],

                True,

                255,

                2

            )


    return canvas

In [ ]:
# ======================================================
# Render
# ======================================================


cad_image = render_geometry(

    GEOMETRY_OBJECTS,

    original.shape

)



plt.figure(

figsize=(8,8)

)

plt.imshow(

cad_image,

cmap="gray"

)

plt.title(

"CAD Render"

)

plt.axis("off")

plt.show()

In [ ]:
# ======================================================
# Resize
# ======================================================


def align_images(a,b):


    h,w=a.shape


    b=cv2.resize(

        b,

        (w,h)

    )


    return a,b



original,cad_image=align_images(

    original,

    cad_image

)

In [ ]:
# ======================================================
# SSIM
# ======================================================


def calculate_similarity(

    img1,

    img2

):


    score,_=ssim(

        img1,

        img2,

        full=True

    )


    return score



score=calculate_similarity(

    original,

    cad_image

)



print(

"Similarity:",

round(score*100,2),

"%"

)

In [ ]:
# ======================================================
# Difference
# ======================================================


difference=cv2.absdiff(

    original,

    cad_image

)



plt.figure(

figsize=(8,8)

)

plt.imshow(

difference,

cmap="gray"

)

plt.title(

"Difference"

)

plt.axis("off")

plt.show()

In [ ]:
# ======================================================
# AI Controller
# ======================================================


OPT_CONFIG={


"target":0.99,


"max_iteration":20,


"epsilon":0.01,


"min_area":50


}



OPT_CONFIG

In [ ]:
# ======================================================
# Auto Adjust
# ======================================================


def adjust(score):


    if score < 0.80:


        OPT_CONFIG["epsilon"]*=0.8


        OPT_CONFIG["min_area"]+=20



    elif score <0.95:


        OPT_CONFIG["epsilon"]*=0.9



    else:


        OPT_CONFIG["epsilon"]*=0.98



    return OPT_CONFIG

In [ ]:
# ======================================================
# Feedback Loop
# ======================================================


history=[]


best_score=0



for i in range(

    OPT_CONFIG["max_iteration"]

):


    print(

    "Iteration",

    i+1

    )


    score=calculate_similarity(

        original,

        cad_image

    )


    history.append(

        score

    )


    print(

    round(score*100,2),

    "%"

    )



    if score > best_score:


        best_score=score



    if score >= OPT_CONFIG["target"]:


        print(

        "TARGET REACHED"

        )

        break



    adjust(score)

In [ ]:
# ======================================================
# Save Report
# ======================================================


REPORT={


"best_similarity":

best_score,


"iterations":

len(history),


"history":

history,


"status":

"SUCCESS"

}



REPORT_FILE=(

ROOT /

"output" /

"AI_Optimization_Report.json"

)



with open(

REPORT_FILE,

"w"

) as f:


    json.dump(

        REPORT,

        f,

        indent=4

    )


print(

REPORT_FILE

)

In [ ]:
PROJECT_STATE.update({

    "optimization_report":

    str(REPORT_FILE),


    "similarity":

    best_score

})



with open(

STATE_FILE,

"w",

encoding="utf-8"

) as f:


    json.dump(

        PROJECT_STATE,

        f,

        indent=4

    )



print(
"Feedback Complete"
)